# পাঠ ১৩ - এজেন্ট মেমোরি


## সেটআপ

এই নোটবুকটি দেখাবে কীভাবে **Microsoft Agent Framework** (MAF) ব্যবহার করে **স্থায়ী মেমরি** সহ একটি ট্রাভেল বুকিং এজেন্ট তৈরি করবেন।

আপনি শিখবেন কীভাবে বিভিন্ন ধরনের এজেন্ট মেমরি — ওয়ার্কিং, ক্ষণস্থায়ী, এবং দীর্ঘস্থায়ী — একটি এজেন্টের কথোপকথনের মাঝে তথ্য ধরে রাখা এবং ব্যবহার করার ওপর প্রভাব ফেলে।

**পূর্বশর্ত:**
- একটি Microsoft Foundry প্রকল্প যেখানে একটি ডিপ্লয় করা চ্যাট মডেল রয়েছে (যেমন `gpt-5-mini`)।
- Azure CLI দিয়ে লগইন করা হয়েছে — আপনার টার্মিনালে `az login` চালান।
- `AZURE_AI_PROJECT_ENDPOINT` — আপনার Microsoft Foundry প্রকল্পের এন্ডপয়েন্ট।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## এজেন্ট মেমোরির প্রকারভেদ

AI এজেন্ট বিভিন্ন ধরনের মেমोरी ব্যবহার করতে পারে, যা প্রতিটি আলাদা উদ্দেশ্য পূরণ করে:

### ওয়ার্কিং মেমোরি
কথোপকথনের থ্রেড নিজেই — একটি সেশনে বিনিময়িত বার্তা। এজেন্ট একই থ্রেডে পূর্ববর্তী বার্তাগুলোর দিকে ফিরে দেখতে পারে যাতে সংগতিশীলতা বজায় থাকে। MAF-এ এটি **`agent.create_session()`** এর মাধ্যমে তৈরি হয়, যা একটি `AgentSession` প্রদান করে।

### শর্ট-টার্ম মেমোরি
এমন তথ্য যা একটি কাজ বা সেশনের সময়কাল ধরে থাকে কিন্তু স্থায়ীভাবে সংরক্ষিত হয় না। উদাহরণস্বরূপ, এজেন্ট একটি বহু-চক্রের পরিকল্পনা কথোপকথনের সময় তথ্য সংগ্রহ করতে পারে এবং সেগুলো ব্যবহার করে চূড়ান্ত পরিকল্পনা তৈরি করে।

### লং-টার্ম মেমোরি
এমন প্রেফারেন্স এবং তথ্য যা **সেশন জুড়ে** টিকে থাকে। পুনরায় আসা ব্যবহারকারী তাদের খাদ্যসংক্রান্ত বিধিনিষেধ বা ভ্রমণ শৈলী পুনরাবৃত্তি করতে হবেনা। লং-টার্ম মেমোরি সাধারণত একটি বাহ্যিক স্টোর দ্বারা ব্যাক-আপ করা হয় — একটি ডাটাবেস, ফাইল, বা ভেক্টর ইনডেক্স — এবং টুলসের মাধ্যমে এজেন্টের সামনে প্রদর্শিত হয়।


## সেশন সহ ওয়ার্কিং মেমরি

স্মৃতির সবচেয়ে সহজ রূপ হল কথোপকথন সেশন। যখন আপনি একই সেশন (যেটি `agent.create_session()` দ্বারা তৈরি হয়) ক্রমান্বয়ে `agent.run()` কলগুলিতে পাঠান, তখন এজেন্ট সেই কথোপকথনের পূর্ণ ইতিহাস দেখতে পায় এবং আগের বিবরণগুলি মনে রাখতে পারে।

চলুন একটি ভ্রমণ এজেন্ট তৈরি করি এবং ওয়ার্কিং মেমরি প্রদর্শন করি।


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

এজেন্ট সঠিকভাবে বাজেটটি স্মরণ করেছিল কারণ দুইটি বার্তাই একই সেশন শেয়ার করে। এটিই হলো **কার্যকর স্মৃতি** — এটি শুধুমাত্র সেশনের জীবদ্দশার জন্য বিদ্যমান থাকে।

### নতুন থ্রেডে কী ঘটে?

যদি আমরা একটি **নতুন** সেশন শুরু করি, তাহলে এজেন্টের পূর্বের কথোপকথনের কোনো স্মৃতি থাকে না:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## দীর্ঘমেয়াদী স্মৃতি প্যাটার্ন

ব্যবহারকারীর পছন্দগুলি **সেশন জুড়ে** মনে রাখার জন্য, আমাদের একটি স্থায়ী স্টোর দরকার যা কথোপকথনের থ্রেডের বাইরে থাকে। এজেন্ট এই স্টোরটি এক্সেস করে **টুলস** এর মাধ্যমে — ফাংশন যা এটি তথ্য সেভ এবং পুনরুদ্ধারের জন্য কল করতে পারে।

নিচে আমরা একটি সরল ইন-মেমরি পছন্দ স্টোর বাস্তবায়ন করেছি (প্রোডাকশনে আপনি এটি একটি ডাটাবেস বা ভেক্টর ইনডেক্স দ্বারা সাপোর্ট করবেন) এবং এটিকে টুলস হিসেবে প্রকাশ করেছি যা এজেন্ট ব্যবহার করতে পারে।

### আর্কিটেকচার
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### দৃশ্য ১ — প্রথমবারের মতো ব্যবহারকারী বার্ষিকী ভ্রমণ বুক করেন

সারা প্রথমবার এখানে এসেছেন। এজেন্টকে টুলসের মাধ্যমে তার পছন্দ সংরক্ষণ করতে হবে এবং সেগুলো ব্যবহার করে হোটেল সুপারিশ করতে হবে।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### দৃশ্যপট ২ — সারা কয়েক সপ্তাহ পরে ফিরে আসে

সারা একটি **সম্পূর্ণ নতুন থ্রেড** শুরু করে (একটি নতুন সেশন অনুকরণ করছে)। ওয়ার্কিং মেমোরি খালি, তবে দীর্ঘমেয়াদী পছন্দ সংরক্ষণ এখনও তার তথ্য ধারণ করে। এজেন্টটি এটি পুনরুদ্ধার করা এবং সুপারিশগুলিকে ব্যক্তিগতকৃত করতে এটি ব্যবহার করা উচিত।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারসংক্ষেপ

এই পাঠে আপনি এজেন্ট মেমোরির তিনটি প্রকার এবং কীভাবে Microsoft Agent Framework দিয়ে এগুলি বাস্তবায়ন করবেন তা শিখেছেন:

| মেমোরি টাইপ | MAF মেকানিজম | আয়ু |
|---|---|---|
| **ওয়ার্কিং** | `agent.create_session()` | একক কথোপকথন |
| **স্বল্পমেয়াদী** | থ্রেডের মধ্যে সঞ্চিত প্রসঙ্গ | একক কাজ / সেশন |
| **দীর্ঘমেয়াদী** | বাহ্যিক স্টোর যা `@tool` ফাংশনের মাধ্যমে অ্যাক্সেস করা হয় | সেশন জুড়ে |

### মূল পয়েন্টসমূহ
1. **`agent.create_session()`** ওয়ার্কিং মেমোরি প্রদান করে — এজেন্ট একটি সেশনের মধ্যে সম্পূর্ণ কথোপকথনের ইতিহাস দেখতে পায়।
2. **নতুন সেশনগুলো প্রসঙ্গ হারায়** — দীর্ঘমেয়াদী মেমোরি ছাড়া এজেন্ট অতীত কথোপকথন মনে রাখতে পারে না।
3. **`@tool` ফাংশনগুলো** এই দূরত্ব পেরিয়ে দেয় — এগুলো এজেন্টকে একটি টেকসই স্টোর থেকে তথ্য সংরক্ষণ ও আহরণ করতে দেয়।
4. **ব্যক্তিগতকরণ সময়ের সঙ্গে উন্নত হয়** — যত বেশি পছন্দসমূহ সংরক্ষিত হয়, এজেন্টের সুপারিশ তত ভালো হয়।

### বাস্তব জীবনের প্রয়োগসমূহ
- **গ্রাহক সেবা**: গ্রাহকের ইতিহাস ও পছন্দ মনে রাখা
- **ব্যক্তিগত সহকারী**: দিন বা সপ্তাহ জুড়ে প্রসঙ্গ বজায় রাখা
- **স্বাস্থ্যসেবা**: রোগীর তথ্য ও পছন্দ ট্র্যাক করা
- **ই-কমার্স**: ইতিহাসের ভিত্তিতে ব্যক্তিগতকৃত কেনাকাটা

### পরবর্তী পদক্ষেপসমূহ
- ইন-মেমোরি ডিক্টের পরিবর্তে একটি ডাটাবেস বা ভেক্টর স্টোর ব্যবহার করুন (যেমন Azure AI সার্চ)
- সময়-সংবেদনশীল তথ্যের জন্য মেমোরি মেয়াদ শেষ করার ব্যবস্থা যোগ করুন
- শেয়ার করা মেমোরি সহ বহু-এজেন্ট সিস্টেম তৈরি করুন
- নলেজ-গ্রাফ-ব্যাকড মেমোরির জন্য Cognee নোটবুকটি অন্বেষণ করুন


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
